# 05c · Object Detection

Classification asks *"what is it?"*  
Detection asks *"what is it AND where is it?"*

This notebook covers bounding boxes, IoU, pretrained detection models, and NMS.

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.utils import draw_bounding_boxes
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont

plt.rcParams["figure.figsize"] = (10, 6)

## Classification vs Detection

| Task | Input | Output |
|------|-------|--------|
| **Classification** | Image | Single label: "dog" |
| **Detection** | Image | Multiple labels + bounding boxes: "dog" at (50,30,200,180) |
| **Segmentation** | Image | Pixel-level masks for each object |

Detection is harder because the model must find *all* objects and localize each one.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

scene = Image.new("RGB", (300, 200), color=(135, 206, 235))
draw = ImageDraw.Draw(scene)
draw.rectangle([30, 80, 120, 170], fill=(139, 69, 19))
draw.ellipse([55, 40, 95, 80], fill=(0, 128, 0))
draw.rectangle([180, 100, 270, 170], fill=(220, 20, 60))
draw.ellipse([230, 30, 280, 80], fill=(255, 215, 0))

ax1.imshow(scene)
ax1.set_title("Classification: \"outdoor scene\"", fontsize=13)
ax1.axis("off")

ax2.imshow(scene)
for (box, label, color) in [
    ([30, 40, 120, 170], "tree", "lime"),
    ([180, 30, 280, 170], "house", "red"),
]:
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                              linewidth=2, edgecolor=color, facecolor="none")
    ax2.add_patch(rect)
    ax2.text(box[0], box[1]-5, label, color=color, fontsize=12, fontweight="bold")
ax2.set_title("Detection: \"tree\" at [...] + \"house\" at [...]", fontsize=13)
ax2.axis("off")
plt.tight_layout()
plt.show()

---
## Bounding Boxes

Two common formats:

| Format | Values | Used By |
|--------|--------|---------|
| `(x1, y1, x2, y2)` | Top-left and bottom-right corners | PyTorch, COCO |
| `(x, y, w, h)` | Top-left corner + width and height | YOLO, some APIs |

```
(x1, y1) ─────────────┐
│                      │
│      Object          │  height = y2 - y1
│                      │
└─────────────(x2, y2)
       width = x2 - x1
```

In [ ]:
canvas = Image.new("RGB", (400, 300), color=(240, 240, 240))
draw = ImageDraw.Draw(canvas)

draw.rectangle([50, 50, 180, 200], fill=(100, 149, 237))
draw.rectangle([220, 80, 350, 250], fill=(255, 165, 0))
draw.ellipse([140, 20, 260, 100], fill=(50, 205, 50))

boxes = [
    {"coords": [50, 50, 180, 200], "label": "Blue Box", "color": "blue"},
    {"coords": [220, 80, 350, 250], "label": "Orange Box", "color": "darkorange"},
    {"coords": [140, 20, 260, 100], "label": "Green Circle", "color": "green"},
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(canvas)
for b in boxes:
    x1, y1, x2, y2 = b["coords"]
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=2.5, edgecolor=b["color"], facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-8, f"{b['label']} ({x1},{y1},{x2},{y2})",
            color=b["color"], fontsize=10, fontweight="bold")
ax.set_title("Bounding Boxes: (x1, y1, x2, y2) format")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## IoU — Intersection over Union

IoU measures how well a predicted box matches the ground truth.  
It's the standard metric for bounding box quality.

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

- IoU = 1.0 → perfect overlap
- IoU = 0.0 → no overlap
- IoU ≥ 0.5 → typically considered a "correct" detection

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

test_cases = [
    ([50, 50, 150, 150], [80, 80, 180, 180], "Partial overlap"),
    ([50, 50, 150, 150], [50, 50, 150, 150], "Perfect overlap"),
    ([50, 50, 100, 100], [200, 200, 300, 300], "No overlap"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (b1, b2, title) in zip(axes, test_cases):
    iou_val = compute_iou(b1, b2)

    ax.set_xlim(0, 350)
    ax.set_ylim(350, 0)
    ax.set_aspect("equal")

    r1 = patches.Rectangle((b1[0], b1[1]), b1[2]-b1[0], b1[3]-b1[1],
                            linewidth=2, edgecolor="blue", facecolor="blue", alpha=0.3)
    r2 = patches.Rectangle((b2[0], b2[1]), b2[2]-b2[0], b2[3]-b2[1],
                            linewidth=2, edgecolor="red", facecolor="red", alpha=0.3)
    ax.add_patch(r1)
    ax.add_patch(r2)
    ax.set_title(f"{title}\nIoU = {iou_val:.2f}", fontsize=12)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---
## YOLO — You Only Look Once

The key insight of YOLO: treat detection as a **single regression problem**.  
Instead of scanning the image with sliding windows or region proposals,  
divide the image into a grid and predict everything in one forward pass.

```
┌────┬────┬────┐     Each grid cell predicts:
│    │    │ 🐕 │     • B bounding boxes (x, y, w, h, confidence)
├────┼────┼────┤     • C class probabilities
│    │ 🚗 │    │
├────┼────┼────┤     One forward pass = all detections
│    │    │    │     → Very fast (real-time!)
└────┴────┴────┘
```

YOLO has gone through many versions (v1–v8+), each faster and more accurate.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
grid_img = Image.new("RGB", (300, 300), color=(245, 245, 245))
ax.imshow(grid_img)

grid_size = 7
cell_w, cell_h = 300 / grid_size, 300 / grid_size

for i in range(grid_size + 1):
    ax.axhline(y=i * cell_h, color="gray", linewidth=0.5, alpha=0.5)
    ax.axvline(x=i * cell_w, color="gray", linewidth=0.5, alpha=0.5)

detections = [
    {"box": [30, 40, 160, 200], "label": "dog", "conf": 0.92, "cell": (2, 3), "color": "blue"},
    {"box": [170, 60, 280, 180], "label": "car", "conf": 0.87, "cell": (5, 3), "color": "red"},
    {"box": [100, 200, 200, 290], "label": "cat", "conf": 0.78, "cell": (3, 6), "color": "green"},
]

for det in detections:
    x1, y1, x2, y2 = det["box"]
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=2.5, edgecolor=det["color"], facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{det['label']} {det['conf']:.0%}",
            color=det["color"], fontsize=11, fontweight="bold")

    cx, cy = det["cell"]
    cell_rect = patches.Rectangle((cx * cell_w, cy * cell_h), cell_w, cell_h,
                                   linewidth=2, edgecolor=det["color"],
                                   facecolor=det["color"], alpha=0.15)
    ax.add_patch(cell_rect)

ax.set_title("YOLO: Grid cells predict bounding boxes", fontsize=14)
ax.set_xlim(0, 300)
ax.set_ylim(300, 0)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Using a Pretrained Detection Model

torchvision includes several pretrained detection models.  
Let's use **Faster R-CNN** (ResNet-50 + FPN backbone) trained on COCO (80 object classes).

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights)
model.eval()

coco_labels = weights.meta["categories"]
print(f"Model detects {len(coco_labels)} classes")
print(f"First 20: {coco_labels[:20]}")

In [ ]:
scene = Image.new("RGB", (640, 480), color=(135, 200, 235))
draw = ImageDraw.Draw(scene)

draw.rectangle([50, 200, 200, 400], fill=(160, 82, 45))
draw.polygon([(50, 200), (125, 120), (200, 200)], fill=(139, 0, 0))
draw.rectangle([100, 280, 150, 400], fill=(101, 67, 33))
draw.rectangle([60, 240, 90, 270], fill=(173, 216, 230))
draw.rectangle([160, 240, 190, 270], fill=(173, 216, 230))

draw.rectangle([300, 250, 500, 420], fill=(70, 70, 70))
draw.ellipse([320, 360, 380, 420], fill=(30, 30, 30))
draw.ellipse([420, 360, 480, 420], fill=(30, 30, 30))
draw.rectangle([280, 280, 520, 350], fill=(200, 0, 0))
draw.rectangle([300, 290, 370, 340], fill=(173, 216, 230))
draw.rectangle([430, 290, 500, 340], fill=(173, 216, 230))

draw.ellipse([550, 20, 620, 90], fill=(255, 255, 0))

draw.rectangle([0, 400, 640, 480], fill=(34, 139, 34))

preprocess = weights.transforms()
input_tensor = preprocess(scene)

with torch.no_grad():
    predictions = model([input_tensor])

pred = predictions[0]
print(f"Detected {len(pred['boxes'])} objects")
for i in range(min(5, len(pred["boxes"]))):
    label = coco_labels[pred["labels"][i]]
    score = pred["scores"][i].item()
    box = pred["boxes"][i].tolist()
    print(f"  {label}: {score:.2%} at [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")

In [ ]:
def draw_detections(image, prediction, labels, threshold=0.5):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    ax1.imshow(image)
    ax1.set_title("Original Image")
    ax1.axis("off")

    ax2.imshow(image)
    colors = plt.cm.Set1(np.linspace(0, 1, 10))

    kept = 0
    for i in range(len(prediction["boxes"])):
        score = prediction["scores"][i].item()
        if score < threshold:
            continue

        box = prediction["boxes"][i].tolist()
        label_idx = prediction["labels"][i].item()
        label = labels[label_idx]
        color = colors[kept % len(colors)]

        rect = patches.Rectangle(
            (box[0], box[1]), box[2]-box[0], box[3]-box[1],
            linewidth=2.5, edgecolor=color, facecolor="none"
        )
        ax2.add_patch(rect)
        ax2.text(box[0], box[1]-8, f"{label} {score:.0%}",
                color="white", fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8))
        kept += 1

    ax2.set_title(f"Detections (threshold={threshold}, found={kept})")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()

draw_detections(scene, pred, coco_labels, threshold=0.3)

---
## Non-Max Suppression (NMS)

A detector often produces **multiple overlapping boxes** for the same object.  
NMS keeps only the best one:

1. Sort all boxes by confidence score (highest first)
2. Take the top box → add to final results
3. Remove all remaining boxes with IoU > threshold against the kept box
4. Repeat until no boxes remain

PyTorch has a built-in: `torchvision.ops.nms(boxes, scores, iou_threshold)`

In [ ]:
overlapping_boxes = torch.tensor([
    [100.0, 100.0, 250.0, 280.0],
    [110.0, 105.0, 260.0, 285.0],
    [105.0, 95.0, 245.0, 275.0],
    [120.0, 110.0, 270.0, 290.0],
    [350.0, 100.0, 480.0, 250.0],
    [355.0, 105.0, 485.0, 255.0],
    [345.0, 95.0, 475.0, 245.0],
])
scores = torch.tensor([0.9, 0.75, 0.8, 0.65, 0.95, 0.7, 0.6])

keep = torchvision.ops.nms(overlapping_boxes, scores, iou_threshold=0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors_before = plt.cm.Reds(np.linspace(0.3, 0.9, len(overlapping_boxes)))

ax1.set_xlim(0, 550)
ax1.set_ylim(350, 0)
ax1.set_aspect("equal")
for i, (box, score) in enumerate(zip(overlapping_boxes, scores)):
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                              linewidth=2, edgecolor=colors_before[i], facecolor="none")
    ax1.add_patch(rect)
    ax1.text(box[0], box[1]-5, f"{score:.0%}", fontsize=9, color=colors_before[i])
ax1.set_title(f"Before NMS: {len(overlapping_boxes)} boxes", fontsize=13)
ax1.set_facecolor("#f5f5f5")

ax2.set_xlim(0, 550)
ax2.set_ylim(350, 0)
ax2.set_aspect("equal")
kept_colors = ["blue", "green", "red", "purple"]
for j, idx in enumerate(keep):
    box = overlapping_boxes[idx]
    score = scores[idx]
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                              linewidth=3, edgecolor=kept_colors[j % len(kept_colors)],
                              facecolor=kept_colors[j % len(kept_colors)], alpha=0.2)
    ax2.add_patch(rect)
    ax2.text(box[0], box[1]-5, f"{score:.0%}", fontsize=11,
            color=kept_colors[j % len(kept_colors)], fontweight="bold")
ax2.set_title(f"After NMS: {len(keep)} boxes kept", fontsize=13)
ax2.set_facecolor("#f5f5f5")

plt.suptitle("Non-Max Suppression: Remove Redundant Detections", fontsize=14)
plt.tight_layout()
plt.show()

---
## Detection Architectures Overview

| Family | Speed | Accuracy | Key Idea |
|--------|-------|----------|----------|
| **R-CNN → Fast R-CNN → Faster R-CNN** | Slow → Fast | High | Region proposals + classification |
| **SSD** (Single Shot Detector) | Fast | Medium | Multi-scale predictions from feature maps |
| **YOLO** (v5, v7, v8) | Very Fast | High | Grid-based single-pass prediction |
| **DETR** | Medium | High | Transformer-based, end-to-end, no anchors |
| **RT-DETR** | Fast | High | Real-time DETR variant |

**Practical guidance**:
- Need real-time detection? → **YOLOv8** or **RT-DETR**
- Need best accuracy? → **Faster R-CNN** or **DETR** (with a big backbone)
- Just learning / prototyping? → **torchvision's Faster R-CNN** (easiest API)

---
### Key Takeaways

| Concept | Remember |
|---|---|
| Bounding box | `(x1, y1, x2, y2)` = top-left to bottom-right |
| IoU | Intersection / Union — ≥0.5 is usually "correct" |
| YOLO | Grid-based, single forward pass → real-time |
| Faster R-CNN | Region proposals → more accurate, slower |
| NMS | Remove duplicate detections, keep the most confident |
| Pretrained models | `torchvision.models.detection` — COCO-trained, 80 classes |